# 05 Morph evoked source estimates to fsaverage

This notebook morphs saved native-subject evoked source estimates to a common FreeSurfer subject, usually `fsaverage`.

Inputs per recording/condition:

- native evoked source estimates from `04_apply_inverse_evokeds.ipynb`,
- FreeSurfer subjects directory containing the individual subject and `fsaverage`.

Outputs are written to `derivatives/meeg-pipeline/sub-*/meg/source_estimates_fsaverage/`.


## Setup

In [ ]:
from __future__ import annotations

from pathlib import Path

import mne
import pandas as pd
from IPython.display import display

from meeg_pipeline.config import load_config
from meeg_pipeline.source_modeling import (
    morph_source_estimate_config_to_dataframe,
    morph_source_estimate_input_overview_to_dataframe,
    morph_evoked_source_estimates_for_recordings,
    morphed_source_estimate_results_to_dataframe,
    morphed_source_estimate_qc_to_dataframe,
)
from meeg_pipeline.workflow import (
    existing_output_policy_for_step,
    iter_recordings,
    recordings_to_dataframe,
)

## MNE logging

In [ ]:
mne.set_log_level("WARNING")

## Project config

In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    """Find project root by searching upward for configs/local.yaml."""
    start = Path.cwd() if start is None else Path(start).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "local.yaml").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find project root by searching for configs/local.yaml "
        f"above {start}"
    )


PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "configs" / "local.yaml"
config = load_config(CONFIG_PATH)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CONFIG_PATH:", CONFIG_PATH)

## Run settings

Methodological defaults are read from `configs/local.yaml` under `source.morph`. The flags here only control notebook execution.

In [ ]:
SUBJECTS = "all"
SESSIONS = "all"
TASKS = "all"
RUNS = "all"

OVERWRITE_STEPS = []

RUN_MORPH = True
RUN_QC = True
MAX_QC_ROWS = None

# Useful for a quick smoke test before running the full batch.
MAX_RECORDINGS_FOR_TEST = None

# Optional overrides; leave as None to use configs/local.yaml.
MORPH_METHOD = None
MORPH_SUBJECT_TO = None
MORPH_SPACING = None
MORPH_SMOOTH = None
PICK_CONDITIONS = None

In [ ]:
display(morph_source_estimate_config_to_dataframe(config))

## Select recordings

In [ ]:
selected_recordings = list(
    iter_recordings(
        config,
        subjects=SUBJECTS,
        sessions=SESSIONS,
        tasks=TASKS,
        runs=RUNS,
    )
)

if MAX_RECORDINGS_FOR_TEST is not None:
    selected_recordings = selected_recordings[:MAX_RECORDINGS_FOR_TEST]

recordings_df = recordings_to_dataframe(selected_recordings)
display(recordings_df)

## Input overview

In [ ]:
on_existing = existing_output_policy_for_step("morph_evoked_source_estimates", OVERWRITE_STEPS)

morph_overview = morph_source_estimate_input_overview_to_dataframe(
    config,
    selected_recordings,
    on_existing=on_existing,
    method=MORPH_METHOD,
    pick_conditions=PICK_CONDITIONS,
    subject_to=MORPH_SUBJECT_TO,
)

display(morph_overview)

if not morph_overview.empty:
    display(
        morph_overview
        .groupby(["task", "status"], dropna=False)
        .size()
        .reset_index(name="n")
        .sort_values(["task", "status"])
    )

## Morph source estimates

In [ ]:
if RUN_MORPH:
    morph_results = morph_evoked_source_estimates_for_recordings(
        config,
        selected_recordings,
        on_existing=on_existing,
        method=MORPH_METHOD,
        pick_conditions=PICK_CONDITIONS,
        subject_to=MORPH_SUBJECT_TO,
        spacing=MORPH_SPACING,
        smooth=MORPH_SMOOTH,
        verbose=True,
    )
    morph_results_df = morphed_source_estimate_results_to_dataframe(morph_results)
else:
    print("Skipped source-estimate morphing.")
    morph_results_df = pd.DataFrame()

display(morph_results_df)

if not morph_results_df.empty:
    display(
        morph_results_df
        .groupby(["task", "status"], dropna=False)
        .size()
        .reset_index(name="n")
        .sort_values(["task", "status"])
    )

## QC

In [ ]:
if RUN_QC:
    morph_qc = morphed_source_estimate_qc_to_dataframe(
        config,
        morph_results_df,
        max_rows=MAX_QC_ROWS,
    )
else:
    print("Skipped morphed-source-estimate QC.")
    morph_qc = pd.DataFrame()

display(morph_qc)

## Output reminder

The morphed STCs are intended for vertex-wise group plots, source-space group contrasts, and optional fsaverage-based analyses. Label-time-course extraction can still use native subject space.